# SankhyaVox — GMM Classifier Training

Train and evaluate the GMM baseline on augmented data, test on held-out human speaker.

**Prerequisites:** Run `DataPipeline().build()` and augmentation first.

In [ ]:
# ── stdlib ──
import pickle
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional, Tuple

# ── third-party ──
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## Config

In [ ]:
# ── Paths ──
DATA_PROCESSED = Path("../data_processed")
PROCESSED_DIR  = DATA_PROCESSED          # alias used by SankhyaVoxDataset default
CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Human speaker to hold out for testing
TEST_SPEAKER = "S05"

## SankhyaVoxDataset

**Paste the `SankhyaVoxDataset` class from `dataset/dataset.py` below.**

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  PASTE SankhyaVoxDataset class from dataset/dataset.py HERE         ║
# ╚═══════════════════════════════════════════════════════════════════════╝

## Load Data & Split

In [ ]:
# Training: augmented data, excluding held-out speaker
aug_ds = SankhyaVoxDataset(DATA_PROCESSED, categories=["augmented"])
print(f"Full augmented: {repr(aug_ds)}")
print(f"Augmented speakers: {aug_ds.speakers}")

# Exclude the test speaker's augmented variant
aug_test_speaker = f"aug{TEST_SPEAKER}"  # e.g. "augS05"
train_ds = aug_ds.exclude_speakers([aug_test_speaker])
X_train, y_train = train_ds.get_Xy()
print(f"\nTrain: {len(X_train)} samples (excluded {aug_test_speaker})")

# Testing: real human recordings for held-out speaker
human_ds = SankhyaVoxDataset(DATA_PROCESSED, categories=["human"])
test_ds = human_ds.filter(speaker=TEST_SPEAKER)
X_test, y_test = test_ds.get_Xy()
print(f"Test:  {len(X_test)} samples (human {TEST_SPEAKER})")
print(f"Labels in test: {sorted(set(y_test))}")
print(f"Feature shape example: {X_train[0].shape}")

---
## GMM Classifier

**Paste the `GMMClassifier` class from `models/gmm_classifier.py` below.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  PASTE GMMClassifier class from models/gmm_classifier.py HERE  ║
# ╚══════════════════════════════════════════════════════════════════╝

## Train

In [ ]:
gmm = GMMClassifier(n_components=8)
gmm.fit(X_train, y_train)
print("Training complete.")

## Save

In [ ]:
gmm.save(str(CHECKPOINT_DIR / "gmm_classifier.pkl"))

## Load

In [ ]:
gmm_loaded = GMMClassifier(checkpoint_path=str(CHECKPOINT_DIR / "gmm_classifier.pkl"))

## Test

In [ ]:
gmm_preds = gmm_loaded.predict(X_test)
gmm_acc = accuracy_score(y_test, gmm_preds)
print(f"GMM Accuracy: {gmm_acc:.3f}")
print()
print(classification_report(y_test, gmm_preds, zero_division=0))

## Results

In [ ]:
# Token label mapping for display
VALUE_TO_TOKEN = {
    0: "shunya", 1: "eka", 2: "dvi", 3: "tri", 4: "catur",
    5: "pancha", 6: "shat", 7: "sapta", 8: "ashta", 9: "nava",
    10: "dasha", 20: "vimsati", 100: "shata",
}

labels_sorted = sorted(set(y_test))
label_names = [VALUE_TO_TOKEN.get(l, str(l)) for l in labels_sorted]

cm = confusion_matrix(y_test, gmm_preds, labels=labels_sorted)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=label_names, yticklabels=label_names)
ax.set_title(f"GMM Confusion Matrix — Accuracy: {gmm_acc:.3f}")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPer-class accuracy:")
for i, l in enumerate(labels_sorted):
    row_total = cm[i].sum()
    correct = cm[i, i]
    acc = correct / row_total if row_total > 0 else 0
    print(f"  {VALUE_TO_TOKEN.get(l, str(l)):>8s} ({l:>3d}): {correct}/{row_total} = {acc:.2%}")